# 🔍 BOSS Datalake — Script 2: Bronze (Trino) vs Silver (Trino)

### Checks
| # | Check |
|---|-------|
| 1 | Column Count — Bronze vs Silver |
| 2 | Data Type Validation |
| 3 | Row Count (Bronze unique vs Silver) |
| 4 | Deduplication Check (Silver mein har Id sirf ek baar) |
| 5 | Latest ingestedfordate Check (Silver row = Bronze ki latest row) |
| 6 | Pipeline Columns Null Check |
| 7 | NULL % + Garbage Check (Silver column-wise) |
| 8 | True NULL vs False NULL (Id join) |

---
## 📦 Cell 1 — Libraries

In [99]:
import re
import pandas as pd
import urllib3
from decimal import Decimal, InvalidOperation, getcontext
from IPython.display import display
from trino.dbapi import connect as trino_connect
from trino.auth import BasicAuthentication

getcontext().prec = 38
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 100)
print('✅ Libraries loaded!')

✅ Libraries loaded!


---
## ⚙️ Cell 2 — Configuration

In [106]:
# ── Trino ──
TRINO_CONFIG = {
     'host'    : '3.221.185.123',
    'port'    : 8443,
    'user'    : 'shariq.mehmood',
    'password': 'Prometheus-X!',
    'catalog' : 'iceberg',
    'schema'  : 'qa'
}

BRONZE_TABLE   = 'iceberg.qa.bronze_StreetComparison'
SILVER_TABLE   = 'iceberg.qa.silver_StreetComparison'

# Single key:    PRIMARY_KEY = 'Id'
# Composite key: PRIMARY_KEY = ['Id', 'SystemDate']
PRIMARY_KEY    = 'Id'

PARTITION_COL  = 'CreatedDate'
RETENTION_DAYS = 160

# ── Pipeline columns ──
PIPELINE_COLUMNS = []

# ── Sample size ──
NULL_SAMPLE_SIZE = 1000

# ── PK helper — single ya composite dono handle karta hai ──
def get_pk_expression(pk):
    if isinstance(pk, list):
        parts = [f'CAST({p} AS VARCHAR)' for p in pk]
        return " || '_' || ".join(parts)
    return f'CAST({pk} AS VARCHAR)'

PK_COLS = PRIMARY_KEY if isinstance(PRIMARY_KEY, list) else [PRIMARY_KEY]
PK_EXPR = get_pk_expression(PRIMARY_KEY)

print(f'✅ Bronze  : {BRONZE_TABLE}')
print(f'✅ Silver  : {SILVER_TABLE}')
print(f'✅ PK      : {PRIMARY_KEY}')
print(f'✅ PK expr : {PK_EXPR}')
print(f'✅ Pipeline: {PIPELINE_COLUMNS}')

✅ Bronze  : iceberg.qa.bronze_StreetComparison
✅ Silver  : iceberg.qa.silver_StreetComparison
✅ PK      : Id
✅ PK expr : CAST(Id AS VARCHAR)
✅ Pipeline: []


---
## 🔌 Cell 3 — Connection

In [101]:
def get_trino_connection():
    auth = BasicAuthentication(TRINO_CONFIG['user'], TRINO_CONFIG['password'])
    return trino_connect(
        host=TRINO_CONFIG['host'], port=TRINO_CONFIG['port'],
        user=TRINO_CONFIG['user'], auth=auth,
        http_scheme='https', verify=False,
        catalog=TRINO_CONFIG['catalog'], schema=TRINO_CONFIG['schema']
    )

try:
    conn = get_trino_connection()
    cur  = conn.cursor()
    cur.execute('SELECT 1')
    cur.fetchall()
    print('✅ Trino connected!')
except Exception as e:
    print(f'❌ Trino failed: {e}')

✅ Trino connected!


---
## 📋 Cell 4 — Column Count Check

In [102]:
def get_table_columns(table):
    conn = get_trino_connection()
    cur  = conn.cursor()
    cur.execute(f'DESCRIBE {table}')
    rows = cur.fetchall()
    df   = pd.DataFrame(rows, columns=['column_name','data_type','extra','comment'])
    df['column_name'] = df['column_name'].str.lower()
    return df[['column_name','data_type']]

bronze_cols_df = get_table_columns(BRONZE_TABLE)
silver_cols_df = get_table_columns(SILVER_TABLE)

pipeline_set = set(c.lower() for c in PIPELINE_COLUMNS)
bronze_set   = set(bronze_cols_df['column_name'])
silver_set   = set(silver_cols_df['column_name'])
bronze_excl  = bronze_set - pipeline_set
silver_excl  = silver_set - pipeline_set

matched           = bronze_excl & silver_excl
missing_in_silver = bronze_excl - silver_excl
extra_in_silver   = silver_excl - bronze_excl

print('=' * 65)
print('📋 COLUMN COUNT CHECK — BRONZE vs SILVER')
print('=' * 65)
print(f'  Bronze columns (total)      : {len(bronze_set)}')
print(f'  Silver columns (total)      : {len(silver_set)}')
print(f'  Pipeline columns (excluded) : {len(pipeline_set)}')
print(f'  Bronze (excl pipeline)      : {len(bronze_excl)}')
print(f'  Silver (excl pipeline)      : {len(silver_excl)}')
print(f'  ✅ Matched                  : {len(matched)}')
print(f'\n🔴 Missing in Silver ({len(missing_in_silver)}):')
for c in sorted(missing_in_silver): print(f'  -> {c}')
if not missing_in_silver: print('  ✅ None')
print(f'\n🔵 Extra in Silver ({len(extra_in_silver)}):')
for c in sorted(extra_in_silver): print(f'  -> {c}')
if not extra_in_silver: print('  ✅ None')

📋 COLUMN COUNT CHECK — BRONZE vs SILVER
  Bronze columns (total)      : 42
  Silver columns (total)      : 40
  Pipeline columns (excluded) : 0
  Bronze (excl pipeline)      : 42
  Silver (excl pipeline)      : 40
  ✅ Matched                  : 40

🔴 Missing in Silver (2):
  -> ingested_ts
  -> source_table

🔵 Extra in Silver (0):
  ✅ None


---
## 🧪 Cell 5 — Data Type Validation

In [103]:
TRINO_TYPE_MAP = {
    'integer': 'integer', 'bigint': 'bigint', 'smallint': 'smallint',
    'decimal': 'decimal', 'double': 'double', 'varchar': 'varchar',
    'boolean': 'boolean', 'date': 'date', 'timestamp': 'timestamp',
}

def norm_trino(t):
    return TRINO_TYPE_MAP.get(str(t).lower().split('(')[0].strip(),
                               str(t).lower().split('(')[0].strip())

bronze_type_map = dict(zip(bronze_cols_df['column_name'], bronze_cols_df['data_type']))
silver_type_map = dict(zip(silver_cols_df['column_name'], silver_cols_df['data_type']))

print('=' * 85)
print('🧪 DATA TYPE VALIDATION — BRONZE vs SILVER')
print('=' * 85)
print(f'  {"COLUMN":<35} | {"BRONZE TYPE":<22} | {"SILVER TYPE":<22} | STATUS')
print(f'  {"-" * 85}')

dtype_results = []
for col in sorted(matched):
    b_raw  = bronze_type_map.get(col, 'unknown')
    s_raw  = silver_type_map.get(col, 'unknown')
    ok     = norm_trino(b_raw) == norm_trino(s_raw)
    status = '✅ MATCH' if ok else '❌ MISMATCH'
    print(f'  {col:<35} | {b_raw:<22} | {s_raw:<22} | {status}')
    dtype_results.append({'column': col, 'bronze_type': b_raw, 'silver_type': s_raw, 'status': status})

dtype_df = pd.DataFrame(dtype_results)
mm = len(dtype_df[dtype_df['status'] == '❌ MISMATCH'])
print(f'  {"-" * 85}')
print(f'\n  ✅ Match   : {len(dtype_df)-mm}/{len(dtype_df)}')
print(f'  ❌ Mismatch: {mm}/{len(dtype_df)}')

🧪 DATA TYPE VALIDATION — BRONZE vs SILVER
  COLUMN                              | BRONZE TYPE            | SILVER TYPE            | STATUS
  -------------------------------------------------------------------------------------
  accountnumber                       | varchar                | varchar                | ✅ MATCH
  accounttype                         | varchar                | varchar                | ✅ MATCH
  assettype                           | varchar                | varchar                | ✅ MATCH
  breakamount                         | decimal(19,6)          | decimal(19,6)          | ✅ MATCH
  breakqty                            | decimal(19,8)          | decimal(19,8)          | ✅ MATCH
  breaks                              | varchar                | varchar                | ✅ MATCH
  comp                                | varchar                | varchar                | ✅ MATCH
  comparedate                         | timestamp(6)           | timestamp(6)          

---
## 🔢 Cell 6 — Row Count + Dedup Check

In [107]:
def get_table_counts(table, retention_filter=False):
    tr_conn = get_trino_connection()
    cur = tr_conn.cursor()
    if retention_filter:
        cur.execute(f"""
            SELECT COUNT(*), COUNT(DISTINCT {PK_EXPR})
            FROM {table}
            WHERE CAST({PARTITION_COL} AS DATE) <= CURRENT_DATE - INTERVAL '{RETENTION_DAYS}' DAY
        """)
    else:
        cur.execute(f'SELECT COUNT(*), COUNT(DISTINCT {PK_EXPR}) FROM {table}')
    row = cur.fetchone()
    return row[0], row[1]

bronze_total, bronze_unique = get_table_counts(BRONZE_TABLE, retention_filter=True)
silver_total, silver_unique = get_table_counts(SILVER_TABLE, retention_filter=False)
bronze_dups = bronze_total - bronze_unique
silver_dups = silver_total - silver_unique
row_match   = bronze_unique == silver_total
dedup_ok    = silver_dups == 0

print('=' * 65)
print('🔢 ROW COUNT + DEDUP CHECK — BRONZE vs SILVER')
print('=' * 65)
print(f'  Bronze total rows   : {bronze_total}')
print(f'  Bronze duplicates   : {bronze_dups}')
print(f'  Bronze unique rows  : {bronze_unique}')
print()
print(f'  Silver total rows   : {silver_total}')
print(f'  Silver duplicates   : {silver_dups}')
print()
print(f'  Bronze unique == Silver total : {"✅ MATCH" if row_match else "❌ MISMATCH"}')
if not row_match:
    print(f'  Difference                    : {silver_total - bronze_unique:+d}')
print(f'  Silver dedup check            : {"✅ No duplicates" if dedup_ok else f"❌ {silver_dups} duplicates found!"}')

🔢 ROW COUNT + DEDUP CHECK — BRONZE vs SILVER
  Bronze total rows   : 155409286
  Bronze duplicates   : 77704643
  Bronze unique rows  : 77704643

  Silver total rows   : 77704643
  Silver duplicates   : 0

  Bronze unique == Silver total : ✅ MATCH
  Silver dedup check            : ✅ No duplicates


---
## 🔧 Cell 8 — Pipeline Columns Check

In [32]:
conn = get_trino_connection()
cur  = conn.cursor()

print('=' * 65)
print('🔧 PIPELINE COLUMNS CHECK — SILVER')
print('=' * 65)

pipeline_results = []
all_pass = True

for col in PIPELINE_COLUMNS:
    if col.lower() not in silver_set:
        print(f'  ❌ MISSING  | {col:<20} — Silver mein nahi hai!')
        pipeline_results.append({'column': col, 'status': '❌ MISSING', 'null_count': '-', 'total': '-'})
        all_pass = False
        continue
    cur.execute(f"""
        SELECT COUNT(*),
               COUNT(CASE WHEN "{col}" IS NULL
                          OR CAST("{col}" AS VARCHAR) = '' THEN 1 END)
        FROM {SILVER_TABLE}
    """)
    total, nulls = cur.fetchone()
    pct = round((nulls/total)*100, 2) if total > 0 else 0
    if nulls > 0:
        status = '❌ FAIL'
        all_pass = False
        print(f'  ❌ FAIL | {col:<20} | nulls: {nulls}/{total} ({pct}%)')
    else:
        status = '✅ PASS'
        print(f'  ✅ PASS | {col:<20} | nulls: {nulls}/{total}')
    pipeline_results.append({'column': col, 'status': status, 'null_count': nulls, 'total': total})

print()
print('✅ All pipeline columns OK!' if all_pass else '❌ Issues found!')

🔧 PIPELINE COLUMNS CHECK — SILVER
  ❌ MISSING  | ingested_ts          — Silver mein nahi hai!
  ❌ MISSING  | source_table         — Silver mein nahi hai!

❌ Issues found!


---
## 🔬 Cell 9 — NULL % + Garbage Check (Silver column-wise)

In [21]:
def is_garbage(value):
    if not value or str(value).strip() == '':
        return False
    return bool(re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]').search(str(value)))

def fetch_silver_sample():
    conn = get_trino_connection()
    cur  = conn.cursor()
    cur.execute(f'SELECT * FROM {SILVER_TABLE} LIMIT {NULL_SAMPLE_SIZE}')
    rows = cur.fetchall()
    cols = [d[0].lower() for d in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    print(f'✅ Silver sample: {len(df)} rows fetched')
    return df

silver_sample_df = fetch_silver_sample()
check_cols       = [c for c in silver_sample_df.columns if c not in pipeline_set]
total_rows       = len(silver_sample_df)

print('\n' + '=' * 90)
print('🔬 NULL % + GARBAGE CHECK — SILVER')
print('   (Sirf columns with NULLs ya Garbage print honge)')
print('=' * 90)
print(f'  {"COLUMN":<35} | {"NULL COUNT":>10} | {"NULL %":>8} | {"GARBAGE":>8} | STATUS')
print(f'  {"-" * 85}')

null_garbage_results = []
for col in sorted(check_cols):
    vals          = silver_sample_df[col].tolist()
    null_count    = sum(1 for v in vals if pd.isna(v) or str(v).strip() in ['','None','null','nan'])
    garbage_count = sum(1 for v in vals if not pd.isna(v) and is_garbage(str(v)))
    null_pct      = round((null_count / total_rows) * 100, 2) if total_rows > 0 else 0

    if garbage_count > 0:
        status = '❌ GARBAGE'
    elif null_pct >= 50:
        status = f'⚠️  HIGH NULL ({null_pct}%)'
    elif null_count > 0:
        status = f'⚠️  NULLS ({null_pct}%)'
    else:
        status = '✅ PASS'

    if null_count > 0 or garbage_count > 0:
        print(f'  {col:<35} | {null_count:>10} | {null_pct:>7}% | {garbage_count:>8} | {status}')

    null_garbage_results.append({
        'column': col, 'null_count': null_count,
        'null_pct': null_pct, 'garbage_count': garbage_count, 'status': status
    })

ng_df = pd.DataFrame(null_garbage_results)
print(f'  {"-" * 85}')
print(f'  Columns with NULLs   : {len(ng_df[ng_df["null_count"] > 0])}')
print(f'  Columns with Garbage : {len(ng_df[ng_df["garbage_count"] > 0])}')
print(f'  Clean columns        : {len(ng_df[ng_df["status"] == "✅ PASS"])}')

✅ Silver sample: 1000 rows fetched

🔬 NULL % + GARBAGE CHECK — SILVER
   (Sirf columns with NULLs ya Garbage print honge)
  COLUMN                              | NULL COUNT |   NULL % |  GARBAGE | STATUS
  -------------------------------------------------------------------------------------
  accountuniqueidentifier             |        740 |    74.0% |        0 | ⚠️  HIGH NULL (74.0%)
  adjustedcode                        |       1000 |   100.0% |        0 | ⚠️  HIGH NULL (100.0%)
  aecode                              |       1000 |   100.0% |        0 | ⚠️  HIGH NULL (100.0%)
  covuncov                            |       1000 |   100.0% |        0 | ⚠️  HIGH NULL (100.0%)
  createdby                           |       1000 |   100.0% |        0 | ⚠️  HIGH NULL (100.0%)
  currency                            |          2 |     0.2% |        0 | ⚠️  NULLS (0.2%)
  entrytype                           |       1000 |   100.0% |        0 | ⚠️  HIGH NULL (100.0%)
  executiontimedate          

---
## ✅ Cell 10 — True NULL vs False NULL (Id join — Bronze vs Silver)

In [75]:
def fetch_bronze_deduped_sample():
    """Bronze se sample — latest ingestedfordate per Id (deduped)."""
    conn = get_trino_connection()
    cur  = conn.cursor()
    cur.execute(f"""
        SELECT * FROM (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY "{PK_EXPR}"
                                     ORDER BY ingested_ts DESC) as rn
            FROM {BRONZE_TABLE}
        ) WHERE rn = 1
        LIMIT {NULL_SAMPLE_SIZE}
    """)
    rows = cur.fetchall()
    cols = [d[0].lower() for d in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    if 'rn' in df.columns:
        df = df.drop(columns=['rn'])
    print(f'✅ Bronze sample: {len(df)} rows (deduped)')
    return df

def fetch_silver_by_ids(id_list):
    conn = get_trino_connection()
    cur  = conn.cursor()
    ids_str = ', '.join(str(i) for i in id_list)
    cur.execute(f'SELECT * FROM {SILVER_TABLE} WHERE "{PK_EXPR}" IN ({ids_str})')
    rows = cur.fetchall()
    cols = [d[0].lower() for d in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    print(f'✅ Silver sample: {len(df)} rows')
    return df

bronze_sample = fetch_bronze_deduped_sample()
id_list       = bronze_sample[PK_EXPR.lower()].tolist()
silver_sample = fetch_silver_by_ids(id_list)

pk = PK_EXPR.lower()
bronze_sample[pk] = bronze_sample[pk].astype(str).str.strip()
silver_sample[pk] = silver_sample[pk].astype(str).str.strip()
merged = bronze_sample.merge(silver_sample, on=pk, suffixes=('_bronze','_silver'))

print(f'\n   Bronze rows  : {len(bronze_sample)}')
print(f'   Silver rows  : {len(silver_sample)}')
print(f'   Joined rows  : {len(merged)}')
if len(bronze_sample) - len(merged) > 0:
    print(f'   ⚠️  Unmatched : {len(bronze_sample)-len(merged)} (Id Silver mein nahi mila)')

skip     = pipeline_set | {pk, 'ingested_ts'}
chk_cols = [c.replace('_bronze','') for c in merged.columns
            if c.endswith('_bronze') and c.replace('_bronze','') not in skip]

print('\n' + '=' * 95)
print('✅ TRUE NULL vs FALSE NULL — BRONZE vs SILVER (Id join)')
print('  ✅ TRUE NULL  = Bronze mein bhi NULL tha (expected)')
print('  ❌ FALSE NULL = Bronze mein value thi, Silver mein NULL (dedup bug)')
print('  ⏭️  SKIP       = Silver mein koi NULL nahi')
print('=' * 95)
print(f'  {"COLUMN":<35} | {"SILVER NULL":>11} | {"BRONZE NULL":>11} | {"FALSE NULL":>10} | STATUS')
print(f'  {"-" * 90}')

true_null_results = []
false_null_detail = []

for col in sorted(chk_cols):
    bc = col + '_bronze'
    sc = col + '_silver'
    if bc not in merged.columns or sc not in merged.columns:
        continue

    b_vals = merged[bc]
    s_vals = merged[sc]

    silver_null = s_vals.isna() | s_vals.astype(str).str.strip().isin(['','None','null','nan','NaN'])
    bronze_null = b_vals.isna() | b_vals.astype(str).str.strip().isin(['','None','null','nan','NaN'])
    false_null  = silver_null & ~bronze_null

    sn = int(silver_null.sum())
    bn = int(bronze_null.sum())
    fn = int(false_null.sum())

    if sn == 0:
        status = '⏭️  SKIP'
    elif fn > 0:
        status = '❌ FALSE NULL'
        false_rows = merged[false_null][[pk, bc]].copy()
        if fn <= 5:
            for _, r in false_rows.iterrows():
                false_null_detail.append({
                    'column': col,
                    PK_EXPR: r[pk],
                    'bronze_value': str(r[bc])[:50]
                })
        else:
            sample_ids = false_rows[pk].head(5).tolist()
            false_null_detail.append({
                'column': col,
                PK_EXPR: f'{fn} rows — sample: {sample_ids}',
                'bronze_value': 'multiple'
            })
    else:
        status = '✅ TRUE NULL'

    if sn > 0:
        print(f'  {col:<35} | {sn:>11} | {bn:>11} | {fn:>10} | {status}')

    true_null_results.append({
        'column': col, 'silver_null': sn,
        'bronze_null': bn, 'false_null': fn, 'status': status
    })

print(f'  {"-" * 90}')

if false_null_detail:
    print(f'\n❌ FALSE NULL DETAIL:')
    print(f'  {"COLUMN":<35} | {PK_EXPR:<40} | BRONZE VALUE')
    print(f'  {"-" * 90}')
    for r in false_null_detail:
        print(f'  {str(r["column"]):<35} | {str(r[PK_EXPR]):<40} | {str(r["bronze_value"])[:30]}')
else:
    print('\n✅ Koi FALSE NULL nahi — sab nulls Bronze mein bhi the!')

true_null_df  = pd.DataFrame(true_null_results)
false_null_df = pd.DataFrame(false_null_detail) if false_null_detail else pd.DataFrame()

✅ Bronze sample: 1000 rows (deduped)
✅ Silver sample: 485 rows

   Bronze rows  : 1000
   Silver rows  : 485
   Joined rows  : 485
   ⚠️  Unmatched : 515 (Id Silver mein nahi mila)

✅ TRUE NULL vs FALSE NULL — BRONZE vs SILVER (Id join)
  ✅ TRUE NULL  = Bronze mein bhi NULL tha (expected)
  ❌ FALSE NULL = Bronze mein value thi, Silver mein NULL (dedup bug)
  ⏭️  SKIP       = Silver mein koi NULL nahi
  COLUMN                              | SILVER NULL | BRONZE NULL | FALSE NULL | STATUS
  ------------------------------------------------------------------------------------------
  createdby                           |         485 |         485 |          0 | ✅ TRUE NULL
  currency                            |         485 |         485 |          0 | ✅ TRUE NULL
  source                              |           1 |           1 |          0 | ✅ TRUE NULL
  updatedby                           |           1 |           1 |          0 | ✅ TRUE NULL
  updateddate                         |    

---
## 📊 Cell 11 — Final Summary

In [44]:
col_ok      = len(missing_in_silver) == 0 and len(extra_in_silver) == 0
dtype_ok    = len(dtype_df[dtype_df['status'] == '❌ MISMATCH']) == 0
row_ok      = row_match
dedup_ok_f  = dedup_ok
pipeline_ok = all(r['status'] == '✅ PASS' for r in pipeline_results)
garbage_ok  = len(ng_df[ng_df['garbage_count'] > 0]) == 0
false_ok    = len(false_null_df) == 0

checks = [
    ('Column Count',          '✅ PASS' if col_ok      else '❌ FAIL',
     f'Missing: {len(missing_in_silver)}, Extra: {len(extra_in_silver)}'),
    ('Data Types',            '✅ PASS' if dtype_ok    else '❌ FAIL',
     f'Mismatches: {len(dtype_df[dtype_df["status"]=="❌ MISMATCH"])}'),
    ('Row Count',             '✅ PASS' if row_ok      else '❌ FAIL',
     f'Bronze unique: {bronze_unique}, Silver total: {silver_total}'),
    ('Deduplication',         '✅ PASS' if dedup_ok_f  else '❌ FAIL',
     f'Silver duplicates: {silver_dups}'),
    ('Pipeline Columns',      '✅ PASS' if pipeline_ok else '❌ FAIL',
     f'{len(pipeline_results)} columns checked'),
    ('Garbage Values',        '✅ PASS' if garbage_ok  else '❌ FAIL',
     f'Columns with garbage: {len(ng_df[ng_df["garbage_count"] > 0])}'),
    ('True NULL Check',       '✅ PASS' if false_ok    else '❌ FAIL',
     f'False nulls: {len(false_null_df)} columns'),
]

print('=' * 75)
print('📊 FINAL SUMMARY — BRONZE vs SILVER')
print('=' * 75)
print(f'  {"CHECK":<25} | {"STATUS":<12} | DETAIL')
print(f'  {"-" * 70}')
for check, status, detail in checks:
    print(f'  {check:<25} | {status:<12} | {detail}')
print(f'  {"-" * 70}')
print(f'  Bronze : {BRONZE_TABLE}')
print(f'  Silver : {SILVER_TABLE}')
print('=' * 75)
overall = all(s == '✅ PASS' for _, s, _ in checks)
print('\n🎉 ALL CHECKS PASSED!' if overall else '\n❌ SOME CHECKS FAILED — Details upar dekho!')

📊 FINAL SUMMARY — BRONZE vs SILVER
  CHECK                     | STATUS       | DETAIL
  ----------------------------------------------------------------------
  Column Count              | ✅ PASS       | Missing: 0, Extra: 0
  Data Types                | ✅ PASS       | Mismatches: 0
  Row Count                 | ✅ PASS       | Bronze unique: 386299, Silver total: 386299
  Deduplication             | ✅ PASS       | Silver duplicates: 0
  Pipeline Columns          | ❌ FAIL       | 2 columns checked
  Garbage Values            | ✅ PASS       | Columns with garbage: 0
  True NULL Check           | ✅ PASS       | False nulls: 0 columns
  ----------------------------------------------------------------------
  Bronze : iceberg.qa.bronze_taxlot
  Silver : iceberg.qa.silver_taxlot

❌ SOME CHECKS FAILED — Details upar dekho!
